In [1]:
import json
import whisper
from nltk.stem import WordNetLemmatizer
import string

In [2]:
# Load the whisper model once
model = whisper.load_model("small")

In [3]:
# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

In [4]:
isl_dict = {}
try:
    with open("json-validator_.json", "r") as f:
        isl_dict = json.load(f)
        print(f"Loaded ISL dictionary with {len(isl_dict)} entries.")
except FileNotFoundError:
    print("Warning: The 'jsonvalidator.json' file was not found. Dictionary will be empty; signs cannot be mapped.")

Loaded ISL dictionary with 18 entries.


In [5]:
isl_dict2 = {}
try:
    with open("alphabets.json", "r") as f:
        isl_dict2 = json.load(f)
        print(f"Loaded ISL dictionary with {len(isl_dict2)} entries.")
except FileNotFoundError:
    print("Warning: The 'alphabets.json' file was not found. Dictionary will be empty; signs cannot be mapped.")

Loaded ISL dictionary with 26 entries.


In [6]:
def gloss_func(text):
    """Processes text by dropping words, lemmatizing, and reordering for ISL gloss."""
    
    # Input validation
    if not text:
        return ""
    
    text = text.strip().lower()
    words = [w.strip(string.punctuation) for w in text.split()]
    
    # Drop auxiliaries and articles
    drop_words = {"am", "is", "are", "was", "were", "a", "an", "the", "to"}
    words = [w for w in words if w and w not in drop_words and w not in string.punctuation]
    
    # Lemmatize words
    words = [lemmatizer.lemmatize(w, pos="v") for w in words]
    
    # Simple Subject-Verb-Object reorder to Subject-Object-Verb
    if len(words) >= 3:
        subject, verb, obj = words[0], words[1], words[2:]
        gloss = [subject] + obj + [verb]
    else:
        gloss = words
        
    return " ".join(gloss)

In [7]:
def signs(text):
    """Converts a glossed sentence into a sequence of ISL sign image paths."""
    
    # Input validation and lowercase normalization
    text = (text or "").lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    signs = []
    
    if not text:
        return signs
    
    # Try exact match first
    if text in isl_dict:
        return [isl_dict[text]]
    text=gloss_func(text)
    # Word-by-word lookup
    for word in text.split():
        if word in isl_dict:
            signs.append(isl_dict[word])  # word exists -> ISL sign video
        else:
            # fallback: spell with alphabets
            spelled = []
            for ch in word:
                if ch in isl_dict2:
                    spelled.append(isl_dict2[ch])
            if spelled:
                signs.extend(spelled)
            else:
                # if nothing found, append None as placeholder
                signs.append(None)
                    
    return signs

In [8]:
def live_translate(audio_path):
    """
    Main function for translation.
    Transcribes audio and converts it to a sequence of ISL sign images.
    """
    if audio_path is None:
        return []

    try:
        # Use the whisper model to transcribe the audio
        result = model.transcribe(audio_path, task='translate', language='hi')
        transcribed_text = result.get("text", "")
        
        # Process the transcribed text
        print(f"Transcribed: {transcribed_text}")
        glossed = gloss_func(transcribed_text)
        print(f"Glossed: {glossed}")
        sign_sequence = signs(glossed)
        print(f"Sign sequence: {sign_sequence}")
    
    except RuntimeError as e:
        # This catches the specific FFmpeg error
        print(f"Error during audio transcription: {e}")
        return []
    except Exception as e:
        print(f"Unexpected error: {e}")
        return []
        
    return sign_sequence

In [9]:
from PIL import Image

def live_translate_as_animation(audio_path):
   
    image_sequence = live_translate(audio_path) 
    
    if not image_sequence:
        return None
        
    # 2. Open images with Pillow if they are file paths
    pil_images = [Image.open(img) if isinstance(img, str) else img for img in image_sequence]
    
    # 3. Save the sequence as a looping animated GIF
    output_gif_path = "isl_sequence.gif"
    pil_images[0].save(
        output_gif_path,
        save_all=True,
        append_images=pil_images[1:],
        duration=500,  # Time per frame in milliseconds 
        loop=0         # 0 means loop infinitely
    )
    
    return output_gif_path

In [10]:
import gradio as gr
#gradio interface
with gr.Blocks() as demo:
    gr.Markdown("## 🎤 Speech → ISL Sign Sequence")

    with gr.Row():
        audio_in = gr.Audio(sources=["microphone"], type="filepath", label="Speak here")
    
    # A single image component that will play your animated frames like a video
    video_out = gr.Image(label="ISL Sign Animation")

    audio_in.change(
        fn=live_translate_as_animation,
        inputs=audio_in,
        outputs=video_out
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
